In [ ]:
"""Visualize cross-validation, out-of-fold predictions, and LASSO coefficients."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


SCRIPT_DIR = Path(__file__).resolve().parent
RESULT_DIR = SCRIPT_DIR / "hasil_lasso_s20"
PLOT_DIR = RESULT_DIR / "visualisasi"


def main() -> None:
    cv_path = RESULT_DIR / "hasil_cv_alpha.csv"
    prediction_path = RESULT_DIR / "prediksi_oof_lasso.csv"
    coefficient_path = RESULT_DIR / "koefisien_lasso.csv"
    for path in (cv_path, prediction_path, coefficient_path):
        if not path.is_file():
            raise FileNotFoundError(f"Hasil CSV belum ditemukan: {path}")

    cv = pd.read_csv(cv_path, encoding="utf-8-sig").sort_values("alpha")
    predictions = pd.read_csv(prediction_path, encoding="utf-8-sig")
    coefficients = pd.read_csv(coefficient_path, encoding="utf-8-sig")
    PLOT_DIR.mkdir(parents=True, exist_ok=True)

    best = cv.loc[cv["mean_cv_mse"].idxmin()]
    fig, axis = plt.subplots(figsize=(9, 5))
    axis.plot(cv["alpha"], cv["mean_cv_mse"], marker=".", markersize=4)
    axis.axvline(best["alpha"], color="red", linestyle="--", label=f"Terbaik α={best['alpha']:.4g}")
    axis.set(xlabel="Alpha (λ)", ylabel="Mean 5-fold CV MSE", title="Pemilihan penalti LASSO")
    axis.grid(alpha=0.25)
    axis.legend()
    fig.tight_layout()
    fig.savefig(PLOT_DIR / "mse_vs_alpha.png", dpi=160, bbox_inches="tight")

    fig, axis = plt.subplots(figsize=(7, 6))
    actual = predictions["actual_density_cells_per_mL"]
    predicted = predictions["oof_predicted_density_cells_per_mL"]
    axis.scatter(actual, predicted, alpha=0.8)
    low, high = min(actual.min(), predicted.min()), max(actual.max(), predicted.max())
    axis.plot([low, high], [low, high], "r--", label="Prediksi ideal")
    axis.set(
        xlabel="Densitas aktual (sel/mL)",
        ylabel="Prediksi OOF (sel/mL)",
        title="LASSO: aktual vs prediksi OOF",
    )
    axis.grid(alpha=0.25)
    axis.legend()
    fig.tight_layout()
    fig.savefig(PLOT_DIR / "aktual_vs_prediksi_oof.png", dpi=160, bbox_inches="tight")

    ordered = coefficients.sort_values("coefficient_standardized")
    colors = np.where(ordered["coefficient_standardized"] >= 0, "teal", "tomato")
    fig, axis = plt.subplots(figsize=(10, 6))
    axis.barh(ordered["feature"], ordered["coefficient_standardized"], color=colors)
    axis.axvline(0, color="black", linewidth=0.8)
    axis.set(xlabel="Koefisien (fitur terstandardisasi)", title="Koefisien LASSO terpilih")
    axis.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / "koefisien_lasso.png", dpi=160, bbox_inches="tight")

    plt.show()
    plt.close("all")
    print(f"Alpha dengan CV MSE minimum: {best['alpha']:.8g}")
    print(f"Grafik tersimpan di: {PLOT_DIR}")


if __name__ == "__main__":
    main()
